# LightGCN (Biased Evaluation) — Coat Shopping

Baseline LightGCN on the **biased Coat logs** (`train.csv`) with a user-wise 80/20 split.

- Dataset: `../data/coat_data/coat_data/coat/train.csv`
- Evaluation: RMSE/MAE on biased test split
- Output: `biased_results_coat_lightgcn.csv` in this folder.


In [6]:
import os
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Lambda
from tensorflow.keras.models import Model

from sklearn.metrics import mean_squared_error, mean_absolute_error

print("TensorFlow version:", tf.__version__)


def save_biased_results_csv(
    csv_path,
    dataset_name,
    model_arch,
    estimator_name,
    y_true,
    y_pred,
    ndcg5=None,
    ndcg10=None,
    recall5=None,
    recall10=None,
):
    mse = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true, y_pred))

    row = {
        "Dataset": dataset_name,
        "Architecture": model_arch,
        "Model": estimator_name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "NDCG@5": np.nan if ndcg5 is None else float(ndcg5),
        "NDCG@10": np.nan if ndcg10 is None else float(ndcg10),
        "Recall@5": np.nan if recall5 is None else float(recall5),
        "Recall@10": np.nan if recall10 is None else float(recall10),
    }

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])

    df.to_csv(csv_path, index=False)
    print(f"Saved biased results to: {csv_path}")


TensorFlow version: 2.20.0


In [7]:
# =========
# Load biased Coat train split and create user-wise 80/20 split
# =========

# From this notebook's location (biased data/coat shopping), the Coat train.csv
# lives under ../../data/coat_data/coat_data/coat/train.csv inside pranathi3.
ratings = pd.read_csv('../../data/coat_data/coat_data/coat/train.csv')
ratings = ratings[['userId', 'itemId', 'rating']]

train_rows, test_rows = [], []

for user_id, user_data in ratings.groupby('userId'):
    user_data = user_data.sample(frac=1, random_state=42)
    n_items = len(user_data)
    train_size = max(1, int(0.8 * n_items))
    train_rows.append(user_data.iloc[:train_size])
    if train_size < n_items:
        test_rows.append(user_data.iloc[train_size:])

train_df = pd.concat(train_rows)
test_df = pd.concat(test_rows) if test_rows else train_df.sample(frac=0.2, random_state=42)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

user_ids = pd.concat([train_df['userId'], test_df['userId']]).unique()
item_ids = pd.concat([train_df['itemId'], test_df['itemId']]).unique()

user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {i: j for j, i in enumerate(item_ids)}

train_df['user'] = train_df['userId'].map(user2idx)
train_df['item'] = train_df['itemId'].map(item2idx)

test_df['user'] = test_df['userId'].map(user2idx)
test_df['item'] = test_df['itemId'].map(item2idx)

num_users = len(user2idx)
num_items = len(item2idx)

X_train = [train_df['user'].values, train_df['item'].values]
X_test = [test_df['user'].values, test_df['item'].values]

y_train = train_df['rating'].values.astype('float32')
y_test = test_df['rating'].values.astype('float32')

print("Encoded users:", num_users, "items:", num_items)


Train shape: (5510, 3)
Test shape: (1450, 3)
Encoded users: 290 items: 300


In [8]:
# =========
# LightGCN-style baseline model for Coat
# =========

embedding_dim = 32

user_input = Input(shape=(1,), dtype='int32', name='user_input')
item_input = Input(shape=(1,), dtype='int32', name='item_input')

user_embedding = Embedding(num_users, embedding_dim, name='user_embedding')(user_input)
item_embedding = Embedding(num_items, embedding_dim, name='item_embedding')(item_input)

user_vec = Lambda(lambda x: tf.nn.l2_normalize(x, axis=-1))(user_embedding)
item_vec = Lambda(lambda x: tf.nn.l2_normalize(x, axis=-1))(item_embedding)

score = Lambda(lambda x: tf.reduce_sum(x[0] * x[1], axis=-1, keepdims=True))([user_vec, item_vec])

model = Model(inputs=[user_input, item_input], outputs=score)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 32)     │      9,280 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_embedding      │ (None, 1, 32)     │      9,600 │ item_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_3 (Lambda)   │ (None, 1, 32)     │          0 │ user_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_4 (Lambda)   │ (None, 1, 32)     │          0 │ item_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_5 (Lambda)   │ (None, 1, 1)      │          0 │ lambda_3[0][0],   │
│                     │                   │            │ lambda_4[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 18,880 (73.75 KB)

 Trainable params: 18,880 (73.75 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# =========
# Train baseline LightGCN on biased Coat split
# =========

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("C:/Users/prana/OneDrive/Documents/thesis/pranathi3/biased data/coat shopping/lightgcn_coat_best_model.keras", save_best_only=True),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=512,
    callbacks=callbacks,
    verbose=1,
)


Epoch 1/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 8.4457 - mae: 2.5955 - val_loss: 8.8558 - val_mae: 2.6593
Epoch 2/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 8.1146 - mae: 2.5311 - val_loss: 8.8336 - val_mae: 2.6551
Epoch 3/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 7.8513 - mae: 2.4789 - val_loss: 8.7721 - val_mae: 2.6435
Epoch 4/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 7.5898 - mae: 2.4258 - val_loss: 8.6540 - val_mae: 2.6212
Epoch 5/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 7.3192 - mae: 2.3697 - val_loss: 8.4528 - val_mae: 2.5826
Epoch 6/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 7.0198 - mae: 2.3060 - val_loss: 8.1558 - val_mae: 2.5243
Epoch 7/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6.6912 - mae: 2.2336 - val_loss: 7.7584 - val_mae: 2.4442
Epoch 8/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 6.3308 - mae: 2.1517 - val_loss: 7.2843 - val_mae: 2.3452
Epoch 9/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 5.958

In [10]:
def compute_and_save_biased_results_lightgcn_coat():
    """Compute Naive ERM metrics on biased Coat LightGCN split and save to CSV."""
    y_pred = model.predict(X_test, verbose=0).flatten()
    csv_path = "biased_results_coat_lightgcn.csv"
    return save_biased_results_csv(
        csv_path=csv_path,
        dataset_name="COAT",
        model_arch="LightGCN",
        estimator_name="Naive ERM",
        y_true=y_test,
        y_pred=y_pred,
    )

print("Call compute_and_save_biased_results_lightgcn_coat() after training to write CSV.")


Call compute_and_save_biased_results_lightgcn_coat() after training to write CSV.
